# cap10_wf2020_bull_participation_diagnostic_v1

Goal: reconcile the apparent SOXL price/buy-and-hold mismatch in the Main Chart against the checkpoint selector metrics before spending another training loop. This notebook does not promote or retain an actor.

In [1]:
from __future__ import annotations

import json
import math
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, init_qlib
from src.soxl_quant.settings import LOOKBACK, QLIB_DATA_PATH

EXPERIMENT = 'cap10_wf2020_bull_participation_diagnostic_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
SOURCE_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_three_fold_checkpoint_drift_selector_v1'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 20.0
CARD.mkdir(parents=True, exist_ok=True)

WALK_FORWARD_WINDOWS = {
    'wf_2020': {
        'train': ('2010-03-11', '2018-12-31'),
        'valid': ('2019-01-01', '2019-12-31'),
        'test': ('2020-01-01', '2020-12-31'),
    },
    'wf_2022': {
        'train': ('2010-03-11', '2020-12-31'),
        'valid': ('2021-01-01', '2021-12-31'),
        'test': ('2022-01-01', '2022-12-31'),
    },
    'wf_2023_2024': {
        'train': ('2010-03-11', '2022-12-31'),
        'valid': ('2023-01-01', '2023-12-31'),
        'test': ('2024-01-01', '2024-12-31'),
    },
}

def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value

def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n', encoding='utf-8')

def return_pct(start: float, end: float) -> float:
    if not math.isfinite(start) or start == 0:
        raise ValueError('invalid benchmark start price')
    return (float(end) / float(start) - 1.0) * 100.0


In [2]:
profile = load_profile(PROFILE_NAME)
init_qlib(QLIB_DATA_PATH)
candidate_metrics = pd.read_csv(SOURCE_CARD / 'checkpoint_candidate_metrics.csv')
selection_by_fold = pd.read_csv(SOURCE_CARD / 'checkpoint_selection_by_fold.csv')

rows: list[dict[str, Any]] = []
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    dataset = build_dataset(
        train_seg=spec['train'],
        valid_seg=spec['valid'],
        test_seg=spec['test'],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    price_series = dataset.handler.price_frame.droplevel('instrument')['price'].sort_index().loc[spec['test'][0]:spec['test'][1]].dropna()
    if price_series.empty:
        raise ValueError(f'empty price series for {fold_name}')
    tradable_start_idx = min(max(int(LOOKBACK), 0), len(price_series) - 1)
    tradable_end_idx = max(tradable_start_idx, len(price_series) - 2)
    calendar_return = return_pct(float(price_series.iloc[0]), float(price_series.iloc[-1]))
    warmup_aligned_return = return_pct(float(price_series.iloc[tradable_start_idx]), float(price_series.iloc[tradable_end_idx]))

    sel = selection_by_fold[selection_by_fold['fold'].eq(fold_name)].iloc[0]
    selected_candidate = str(sel['selected_candidate'])
    selected = candidate_metrics[
        candidate_metrics['fold'].eq(fold_name) & candidate_metrics['candidate'].eq(selected_candidate)
    ].iloc[0]
    official_bnh = float(selected['test_buy_and_hold_return'])
    official_return = float(selected['test_return'])
    official_relative = float(selected['test_relative_return'])
    calendar_relative = official_return - calendar_return
    warmup_relative = official_return - warmup_aligned_return
    checkpoint_identity_relative = official_return - official_bnh
    checkpoint_relative_identity_gap = official_relative - checkpoint_identity_relative
    rows.append({
        'fold': fold_name,
        'test_start': spec['test'][0],
        'test_end': spec['test'][1],
        'price_rows': int(len(price_series)),
        'calendar_start_date': str(price_series.index[0].date()),
        'calendar_end_date': str(price_series.index[-1].date()),
        'calendar_start_price': float(price_series.iloc[0]),
        'calendar_end_price': float(price_series.iloc[-1]),
        'calendar_buy_and_hold_return': float(calendar_return),
        'tradable_start_date': str(price_series.index[tradable_start_idx].date()),
        'tradable_end_date': str(price_series.index[tradable_end_idx].date()),
        'tradable_start_price': float(price_series.iloc[tradable_start_idx]),
        'tradable_end_price': float(price_series.iloc[tradable_end_idx]),
        'warmup_days': int(tradable_start_idx),
        'warmup_aligned_buy_and_hold_return': float(warmup_aligned_return),
        'checkpoint_selected_candidate': selected_candidate,
        'checkpoint_test_return': official_return,
        'checkpoint_buy_and_hold_return': official_bnh,
        'checkpoint_relative_return': official_relative,
        'calendar_relative_return': float(calendar_relative),
        'warmup_recomputed_relative_return': float(warmup_relative),
        'checkpoint_identity_relative_return': float(checkpoint_identity_relative),
        'checkpoint_relative_identity_gap_pp': float(checkpoint_relative_identity_gap),
        'checkpoint_matches_warmup_benchmark': bool(abs(official_bnh - warmup_aligned_return) < 1e-3),
        'calendar_vs_warmup_bnh_gap_pp': float(warmup_aligned_return - calendar_return),
    })

alignment = pd.DataFrame(rows)
alignment.to_csv(CARD / 'benchmark_alignment.csv', index=False)
alignment


[62689:MainThread](2026-06-13 20:22:23,865) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[62689:MainThread](2026-06-13 20:22:24,268) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[62689:MainThread](2026-06-13 20:22:24,268) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/wf2020-bull-diagnostic/data/qlib/soxl')}


[62689:MainThread](2026-06-13 20:22:25,207) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[62689:MainThread](2026-06-13 20:22:25,207) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


...

[62689:MainThread](2026-06-13 20:22:26,324) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


[62689:MainThread](2026-06-13 20:22:27,644) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[62689:MainThread](2026-06-13 20:22:27,644) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[62689:MainThread](2026-06-13 20:22:27,645) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Init data Done


,fold,test_start,test_end,price_rows,calendar_start_date,calendar_end_date,calendar_start_price,calendar_end_price,calendar_buy_and_hold_return,tradable_start_date,...,checkpoint_selected_candidate,checkpoint_test_return,checkpoint_buy_and_hold_return,checkpoint_relative_return,calendar_relative_return,warmup_recomputed_relative_return,checkpoint_identity_relative_return,checkpoint_relative_identity_gap_pp,checkpoint_matches_warmup_benchmark,calendar_vs_warmup_bnh_gap_pp
0,wf_2020,2020-01-01,2020-12-31,253,2020-01-02,2020-12-31,19.464001,31.098667,59.775308,2020-03-30,...,bc_initialization,46.810017,355.736160,-285.449375,-12.965291,-308.926570,-308.926142,23.476767,True,295.961279
1,wf_2022,2022-01-01,2022-12-31,251,2022-01-03,2022-12-30,72.099998,9.670000,-86.588072,2022-03-30,...,bc_initialization,-6.273652,-76.614257,47.131485,80.314420,70.340616,70.340605,-23.209120,True,9.973804
2,wf_2023_2024,2024-01-01,2024-12-31,252,2024-01-02,2024-12-31,28.040001,27.309999,-2.603429,2024-03-28,...,bc_initialization,-3.575510,-39.952710,38.712278,-0.972081,36.377206,36.377201,2.335077,True,-37.349287


In [3]:
all_match = bool(alignment['checkpoint_matches_warmup_benchmark'].all())
mixed_metric_source = bool(alignment['checkpoint_relative_identity_gap_pp'].abs().max() > 1.0)
wf2020 = alignment.set_index('fold').loc['wf_2020']
large_wf2020_gap = bool(float(wf2020['calendar_vs_warmup_bnh_gap_pp']) > 200.0)

if all_match and mixed_metric_source:
    decision = 'checkpoint_selector_mixed_metric_source_no_promotion'
elif all_match and large_wf2020_gap:
    decision = 'wf2020_benchmark_alignment_explained_no_promotion'
else:
    decision = 'benchmark_alignment_needs_followup_no_promotion'
next_mechanism = (
    'Rerun checkpoint selection with backtest_profile bnh_return and relative_outperformance from the same qlib replay source before any new training.'
    if decision == 'checkpoint_selector_mixed_metric_source_no_promotion'
    else (
        'Keep the checkpoint selector gate on warmup-aligned official metrics, but label dashboard chart returns as calendar-window returns so users do not compare them directly.'
        if decision == 'wf2020_benchmark_alignment_explained_no_promotion'
        else 'Investigate qlib price path and selector metric mismatch before any new training.'
    )
)

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'next_mechanism': next_mechanism,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'cap10_three_fold_checkpoint_drift_selector_v1',
    'reference_baseline': 'soxlbacktest_qlib_buysell_reference',
    'role': 'diagnose',
    'changed_layer': ['model_selection', 'metric_reconciliation', 'dashboard_interpretation'],
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'training_allowed_from_this_card': False,
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'lookback_rows': int(LOOKBACK),
    'all_checkpoint_benchmarks_match_warmup_alignment': all_match,
    'mixed_metric_source_detected': mixed_metric_source,
    'max_checkpoint_relative_identity_gap_pp': float(alignment['checkpoint_relative_identity_gap_pp'].abs().max()),
    'wf2020_calendar_buy_and_hold_return': float(wf2020['calendar_buy_and_hold_return']),
    'wf2020_warmup_aligned_buy_and_hold_return': float(wf2020['warmup_aligned_buy_and_hold_return']),
    'wf2020_benchmark_gap_pp': float(wf2020['calendar_vs_warmup_bnh_gap_pp']),
    'sidecars': ['benchmark_alignment.csv', 'selection.json', 'summary.json', 'conclusion.md'],
}
summary = {
    **selection,
    'rows': alignment.to_dict('records'),
    'conclusion': [
        f"wf_2020 calendar SOXL buy-and-hold is {float(wf2020['calendar_buy_and_hold_return']):.2f}%, while the selector gate benchmark is {float(wf2020['warmup_aligned_buy_and_hold_return']):.2f}% after {int(wf2020['warmup_days'])} warmup rows.",
        f"The checkpoint sidecar buy-and-hold value matches the warmup-aligned replay benchmark: {bool(wf2020['checkpoint_matches_warmup_benchmark'])}.",
        f"However checkpoint relative return is not equal to qlib return minus checkpoint buy-and-hold; max identity gap is {float(alignment['checkpoint_relative_identity_gap_pp'].abs().max()):.2f}pp.",
        'No actor is promoted or retained by this diagnostic notebook.',
    ],
}
write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
print(json.dumps(json_safe(selection), indent=2, allow_nan=False))


{
  "experiment": "cap10_wf2020_bull_participation_diagnostic_v1",
  "created_at": "2026-06-13T12:22:27.659168+00:00",
  "decision": "checkpoint_selector_mixed_metric_source_no_promotion",
  "next_mechanism": "Rerun checkpoint selection with backtest_profile bnh_return and relative_outperformance from the same qlib replay source before any new training.",
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": "cap10_three_fold_checkpoint_drift_selector_v1",
  "reference_baseline": "soxlbacktest_qlib_buysell_reference",
  "role": "diagnose",
  "changed_layer": [
    "model_selection",
    "metric_reconciliation",
    "dashboard_interpretation"
  ],
  "promotion_allowed": false,
  "actor_retention_allowed": false,
  "training_allowed_from_this_card": false,
  "target_relative_return_pp": 30.0,
  "target_max_dd_pct": 20.0,
  "lookback_rows": 60,
  "all_checkpoint_benchmarks_match_warmup_alignment": true,
  "mixed_metric_source_detected": true,
  "

In [4]:
lines = [
    '## Conclusion',
    '',
    f"Decision: `{selection['decision']}`.",
    '',
    'The 2020 chart window and the checkpoint selector use different benchmark anchors:',
    f"- Calendar chart benchmark: {float(wf2020['calendar_start_price']):.4f} -> {float(wf2020['calendar_end_price']):.4f}, return {float(wf2020['calendar_buy_and_hold_return']):.2f}%.",
    f"- Warmup-aligned selector benchmark: {float(wf2020['tradable_start_price']):.4f} -> {float(wf2020['tradable_end_price']):.4f}, return {float(wf2020['warmup_aligned_buy_and_hold_return']):.2f}%.",
    f"- Gap: {float(wf2020['calendar_vs_warmup_bnh_gap_pp']):.2f}pp after {int(wf2020['warmup_days'])} warmup rows.",
    '',
    'The checkpoint benchmark itself matches the warmup-aligned price path, but the checkpoint relative-return field is mixed-source: qlib return minus checkpoint buy-and-hold differs from the saved relative return by up to ' + f"{float(selection['max_checkpoint_relative_identity_gap_pp']):.2f}pp" + '.',
    '',
    'This explains why Main Chart calendar returns can look inconsistent with model-selection relative-return gates. It does not make the failed `wf_2020` gate pass, and it does not promote a model.',
    '',
    f"Next action: {selection['next_mechanism']}",
    '',
    'Artifacts: `benchmark_alignment.csv`, `selection.json`, `summary.json`.',
]
conclusion = '\n'.join(lines).rstrip() + '\n'
(CARD / 'conclusion.md').write_text(conclusion, encoding='utf-8')
display(Markdown(conclusion))


## Conclusion

Decision: `checkpoint_selector_mixed_metric_source_no_promotion`.

The 2020 chart window and the checkpoint selector use different benchmark anchors:
- Calendar chart benchmark: 19.4640 -> 31.0987, return 59.78%.
- Warmup-aligned selector benchmark: 6.7520 -> 30.7713, return 355.74%.
- Gap: 295.96pp after 60 warmup rows.

The checkpoint benchmark itself matches the warmup-aligned price path, but the checkpoint relative-return field is mixed-source: qlib return minus checkpoint buy-and-hold differs from the saved relative return by up to 23.48pp.

This explains why Main Chart calendar returns can look inconsistent with model-selection relative-return gates. It does not make the failed `wf_2020` gate pass, and it does not promote a model.

Next action: Rerun checkpoint selection with backtest_profile bnh_return and relative_outperformance from the same qlib replay source before any new training.

Artifacts: `benchmark_alignment.csv`, `selection.json`, `summary.json`.
